# 6.3 Multi-Agent AI
Multi-Agent AI (MAAI) refers to a system architecture composed of multiple autonomous or semi-autonomous AI-based agents that collaborate — perceiving, reasoning, and acting together — to solve complex, context-rich tasks.

In [ ]:
# %pip install google-genai

## **TASK: Multi-Agent AI with Google Gemini**

In this notebook, you will build a multi-agent AI system to solve math problems using the Google Gemini language model.

Steps:
1. Load the Google Gemini model using the `google-genai` library and set up the API client. You can generate your own API key if you follow these instructions:
   - Create a free student account for Google Gemini under https://gemini.google/students/
   - Go to https://aistudio.google.com/api-keys and copy the Default Gemini API Key
   - Paste the API Key into the cell below
   - The rate limit is specified here: https://ai.google.dev/gemini-api/docs/rate-limits?hl=en
   - You can find further information on API keys here: https://ai.google.dev/gemini-api/docs/api-key?hl=en 
2. Instantiate three agents with different personalities and roles:
   - Planner: understands the math problem and breaks it into steps.
   - Solver: carries out the calculations and proposes a solution.
   - Tutor: checks the solution and explains it clearly for a student.
3. Build a multi-agent workflow to solve math homework:
   - Input: a math word problem or exercise.
   - Output: a step-by-step explanation checked by another agent.

In [ ]:
from pydantic import BaseModel, Field
from google import genai
import ipywidgets as widgets
from IPython.display import display, Markdown

In [ ]:
# 1. Load the Google Gemini model and set up the API client (Create an API key here: https://aistudio.google.com/app/api-keys)
client = genai.Client(api_key="YOUR_API_KEY")
MODEL = "gemini-2.5-flash"


In [ ]:
# Test the Gemini model with a simple prompt
response = client.models.generate_content(
    model=MODEL,
    contents="Explain how AI works in a few words."
)
print(response.text)

In [ ]:
# Define a helper function to call Gemini with a prompt
def call_gemini(prompt: str) -> str:
    # ---- YOUR CODE GOES HERE ---
    return response.text

In [ ]:
# Define the Agent class for different agent roles
class Agent(BaseModel):
    
    name: str = Field(..., description="Name of the agent")
    role_description: str = Field(..., description="Role description of the agent")

    def __call__(self, instruction: str) -> str:

        # ---- YOUR CODE GOES HERE ---

        return call_gemini(prompt)

In [ ]:
# 2. Instantiate Planner, Solver, and Tutor agents with specific role descriptions
# 2.1 Instantiate Planner agent
planner = Agent(
    # ---- YOUR CODE GOES HERE ---
)


In [ ]:
# 2.2 Instantiate Solver agent
solver = Agent(
    # ---- YOUR CODE GOES HERE ---
)

In [ ]:
# 2.3 Instantiate Tutor agent
tutor = Agent(
    # ---- YOUR CODE GOES HERE ---
)

In [ ]:
# 3. Build the multi-agent workflow to solve math homework
def run_math_homework(problem: str) -> dict[str, str]:

    plan_instruction = (
        "Rewrite the math exercise, list knowns and unknowns, "
        "and create a numbered solution plan.\n\n"
        f"Exercise: {problem}"
    )
    plan = planner(plan_instruction)

    solve_instruction = (
        "Here is the student's exercise and a plan. Follow the plan step by step and "
        "compute the final result.\n\n"
        f"Exercise: {problem}\n\nPlan: {plan}"
    )
    solution = solver(solve_instruction)

    tutor_instruction = (
        "Check the solution for correctness and explain it clearly.\n\n"
        f"Exercise: {problem}\n\nPlan: {plan}\n\nSolution: {solution}"
    )
    explanation = tutor(tutor_instruction)

    return {
        "problem": problem,
        "plan": plan,
        "solution": solution,
        "explanation": explanation
    }

In [ ]:
run_math_homework("Solve for x: 3x + 5 = 20")

## **TASK: Interactive Math Interface**

In [ ]:
problem_input = widgets.Textarea(
    value="A train travels at 60 km/h for 2 hours. How far does it go?",
    description="Problem:",
    layout=widgets.Layout(width="100%", height="80px")
)

In [ ]:
run_button = widgets.Button(description="Run solver", button_style="primary")
output_area = widgets.Output()

In [ ]:
def on_run_clicked(b):
    output_area.clear_output()
    with output_area:
        if not problem_input.value.strip():
            print("Please enter a math problem.")
            return
        res = run_math_homework(problem_input.value)

        display(Markdown("### Student Exercise"))
        display(Markdown(res["problem"]))
        display(Markdown("\n---\n"))
        display(Markdown("### Planner Agent"))
        display(Markdown(res["plan"]))
        display(Markdown("\n---\n"))
        display(Markdown("### Solver Agent"))
        display(Markdown(res["solution"]))
        display(Markdown("\n---\n"))
        display(Markdown("### Tutor Agent (Final Explanation)"))
        display(Markdown(res["explanation"]))

run_button.on_click(on_run_clicked)

display(problem_input, run_button, output_area)